# Cochin MS Oo.1.16.2 — PDF → OSIS Converter

Converts the PDF interlinear document into three OSIS XML files:
- `*_hebrew.osis` — Hebrew text only
- `*_hebrew_commented.osis` — Hebrew text with inline footnotes
- `*_translation.osis` — English translation with inline footnotes

**PDF structure per verse:**
1. `Revelation N:V` or `Revelation N:V (Cochin N:AV)` header block
2. Hebrew transcription text (one or more wide text blocks with Hebrew chars)
3. `Hebrew Transcription` label block
4. `Translation: ...` block with English translation
5. Footnotes: blocks starting with `digit space capital-letter`

**Alt numbering:** `Revelation 2:7 (Cochin 2:6)` means standard verse 7, MS verse 6.  
**Empty verses:** `Revelation 2:6 (This verse does not exist...)` → empty placeholder.

In [ ]:
import fitz  # PyMuPDF
import re
import pathlib
from lxml import etree
from collections import defaultdict

# ── Config ──────────────────────────────────────────────────────────────────
PDF_PATH = pathlib.Path("../data/00_source_files/MS_Cochin_Oo.1.16.2_REV_ProjectTruthMinistries.pdf")
OUT_DIR  = pathlib.Path("../data/01_osis")
STEM     = "Cochin_MS_Oo.1.16.2_REV"

# Pages to skip (front matter, back matter). Content starts around page 15 (0-indexed: 14)
FIRST_CONTENT_PAGE = 14   # 0-indexed; page 15 of PDF = Chapter 1 title page

print(f"PDF: {PDF_PATH}")
print(f"Output dir: {OUT_DIR}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

HEBREW_RE = re.compile(r'[֐-׿יִ-ﭏ]')

def has_hebrew(text: str) -> bool:
    return bool(HEBREW_RE.search(text))

def hebrew_char_count(text: str) -> int:
    return sum(1 for c in text if '֐' <= c <= '׿' or 'יִ' <= c <= 'ﭏ')

def is_wide(x0: float, x1: float) -> bool:
    return (x1 - x0) > 50

def clean_hebrew(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_footnote_number(text: str):
    m = re.match(r'^(\d+)\s+([A-Z\'\""].+)', text.strip(), re.DOTALL)
    if m:
        return m.group(1), m.group(2).strip()
    return None

def split_footnote_block(text: str):
    """Split a footnote block into individual footnotes.
    Each starts with a number followed by space and uppercase letter.
    Uses span-level small-font digits in the PDF, but here we split
    the concatenated text by finding footnote-number boundaries.
    """
    # Find all positions where a footnote entry starts
    result = {}
    starts = list(re.finditer(r'(?:^|\.\s+)(\d+)\s+([A-Z\'\""])', text))
    if not starts:
        m = re.match(r'^(\d+)\s+([A-Z\'\""].+)', text.strip(), re.DOTALL)
        if m:
            result[m.group(1)] = m.group(2).strip()
        return result
    for i, match in enumerate(starts):
        n_str = match.group(1)
        start = match.start(1)
        end = starts[i+1].start(1) if i+1 < len(starts) else len(text)
        chunk = text[start:end].strip()
        m = re.match(r'^(\d+)\s+([A-Z\'\""].+)', chunk, re.DOTALL)
        if m:
            result[m.group(1)] = m.group(2).strip()
    return result

def block_texts(block):
    spans = [s for line in block.get("lines", []) for s in line.get("spans", [])]
    if not spans:
        return "", ""
    full_text = "".join(s["text"] for s in spans)
    sizes = [s["size"] for s in spans if s["text"].strip()]
    if not sizes:
        return full_text.strip(), full_text.strip()
    dominant = max(sizes)
    main_text = "".join(s["text"] for s in spans if s["size"] >= dominant * 0.85)
    return full_text.strip(), main_text.strip()

def parse_verse_header(main_text: str):
    main_text = main_text.strip()
    m = re.match(
        r'Revelation\s+(\d+):(\d+[abAB]?)'
        r'(?:\s*\(([^)]+)\))?'
        r'\s*$',
        main_text
    )
    if not m:
        return None
    chapter = int(m.group(1))
    verse   = m.group(2).lower()
    paren   = m.group(3) or ""

    empty  = False
    alt_ch = None
    alt_v  = None

    if "does not exist" in paren.lower():
        empty = True
    elif paren:
        cm = re.match(r'Cochin\s+(\d+):(\d+[abAB]?)', paren)
        if cm:
            alt_ch = int(cm.group(1))
            alt_v  = cm.group(2).lower()

    return {
        "chapter": chapter,
        "verse":   verse,
        "alt_ch":  alt_ch,
        "alt_v":   alt_v,
        "empty":   empty,
        "hebrew":  "",
        "english": "",
        "notes":   {},
        "heb_markers":  [],
        "eng_markers":  [],
    }


# ── RTL-aware Hebrew extraction ─────────────────────────────────────────────

BODY_MARK_RATIO = 0.75

def hebrew_line_segments(block):
    """Per-line RTL reversal. Returns list of lines, each a list of segments."""
    all_lines = []
    for line in block.get("lines", []):
        spans = line.get("spans", [])
        if not spans:
            continue
        sizes = [s["size"] for s in spans if s["text"].strip()]
        if not sizes:
            continue
        max_size = max(sizes)

        sorted_spans = sorted(spans, key=lambda s: s["bbox"][0], reverse=True)

        segments = []
        for s in sorted_spans:
            txt = s["text"]
            sz  = s["size"]
            stripped = txt.strip()
            if not stripped:
                continue
            if sz < max_size * BODY_MARK_RATIO and stripped.isdigit():
                segments.append({"text": "", "marker": int(stripped)})
            else:
                segments.append({"text": txt[::-1], "marker": None})

        if segments:
            all_lines.append(segments)
    return all_lines


def hebrew_text_and_notes(block):
    """Returns (text_str, [(char_offset, footnote_n), ...])"""
    lines = hebrew_line_segments(block)
    result_text = ""
    markers = []

    for i, line_segs in enumerate(lines):
        if i > 0 and result_text:
            result_text += " "
        for seg in line_segs:
            if seg["marker"] is not None:
                markers.append((len(result_text), seg["marker"]))
            else:
                result_text += seg["text"]

    return result_text, markers


# ── English translation span extraction ──────────────────────────────────────

ENG_MARK_RATIO = 0.75

def english_text_and_notes_from_block(block):
    """Returns (text_str, [(char_offset, footnote_n), ...])"""
    spans = [s for line in block.get("lines", []) for s in line.get("spans", [])]
    if not spans:
        return "", []

    sizes = [s["size"] for s in spans if s["text"].strip()]
    if not sizes:
        return "", []
    max_size = max(sizes)

    result_text = ""
    markers = []

    for s in spans:
        txt = s["text"]
        sz  = s["size"]
        stripped = txt.strip()
        if not stripped:
            result_text += txt
            continue
        if sz < max_size * ENG_MARK_RATIO and stripped.isdigit():
            markers.append((len(result_text), int(stripped)))
        else:
            result_text += txt

    return result_text, markers


print("Helper functions defined.")

In [ ]:
# ── PDF parsing ──────────────────────────────────────────────────────────────

VERSE_HEADER_RE = re.compile(r'^Revelation\s+\d+:\d+')
FOOTNOTE_START  = re.compile(r'^\d+\s+[A-Z\'"]')

PAGE_RUNNING_HEADER_Y1 = 48
PAGE_FOOTER_Y0         = 728

FN_MARK_RATIO = 0.75

def is_footnote_block(text: str) -> bool:
    if not FOOTNOTE_START.match(text):
        return False
    total = len(text)
    if total == 0:
        return False
    hc = hebrew_char_count(text)
    if hc / total > 0.15:
        return False
    return True

def extract_footnotes_from_block(block):
    """Use span font sizes to split a footnote block into {n_str: text}."""
    spans = [s for line in block.get("lines", []) for s in line.get("spans", [])]
    if not spans:
        return {}
    sizes = [s["size"] for s in spans if s["text"].strip()]
    if not sizes:
        return {}
    max_size = max(sizes)

    footnotes = {}
    current_n = None
    current_text = ""

    for s in spans:
        txt = s["text"]
        sz = s["size"]
        stripped = txt.strip()
        if not stripped:
            if current_n is not None:
                current_text += txt
            continue
        if sz < max_size * FN_MARK_RATIO and stripped.isdigit():
            if current_n is not None:
                footnotes[current_n] = current_text.strip()
            current_n = stripped
            current_text = ""
        else:
            if current_n is not None:
                current_text += txt
    if current_n is not None:
        footnotes[current_n] = current_text.strip()
    return footnotes

def attach_pending_notes(verse, pending):
    if verse is not None:
        verse["notes"].update(pending)

def _strip_translation_prefix(text, markers):
    """Remove leading whitespace and 'Translation:' prefix, adjusting marker offsets."""
    lstripped = text.lstrip()
    ws_len = len(text) - len(lstripped)
    if ws_len:
        text = lstripped
        markers = [(off - ws_len, n) for off, n in markers if off >= ws_len]
    if text.startswith("Translation:"):
        plen = len("Translation:")
        text = text[plen:]
        markers = [(off - plen, n) for off, n in markers if off >= plen]
    return text.strip(), markers

doc = fitz.open(str(PDF_PATH))
print(f"Opened PDF: {len(doc)} pages")

verses        = []
anomalies     = []
state         = "SEEK"
current       = None
pending_notes = {}
seen_verse_ids = {}

for pg_idx in range(FIRST_CONTENT_PAGE, len(doc)):
    page = doc[pg_idx]
    d    = page.get_text("dict")
    blocks = sorted(d["blocks"], key=lambda b: b["bbox"][1])
    page_footnotes = {}

    for block in blocks:
        if block.get("type") != 0:
            continue
        x0, y0, x1, y1 = block["bbox"]

        if y1 < PAGE_RUNNING_HEADER_Y1 or y0 > PAGE_FOOTER_Y0:
            continue

        full_text, main_text = block_texts(block)
        if not full_text:
            continue

        # ── Footnote detection ──────────────────────────────────────────────
        if is_footnote_block(full_text):
            page_footnotes.update(extract_footnotes_from_block(block))
            continue

        # ── Verse header detection ──────────────────────────────────────────
        if VERSE_HEADER_RE.match(main_text):
            attach_pending_notes(current, pending_notes)
            pending_notes = {}
            parsed = parse_verse_header(main_text)
            if parsed:
                if current is not None:
                    verses.append(current)
                    seen_verse_ids[(current["chapter"], current["verse"])] = len(verses) - 1

                key = (parsed["chapter"], parsed["verse"])
                if not parsed["verse"].endswith(('a','b')) and key in seen_verse_ids:
                    prev_idx = seen_verse_ids.pop(key)
                    prev_v = verses[prev_idx]
                    prev_v["verse"] = prev_v["verse"] + "a"
                    if prev_v["alt_v"] and not prev_v["alt_v"].endswith(('a','b')):
                        prev_v["alt_v"] = prev_v["alt_v"] + "a"
                    seen_verse_ids[(prev_v["chapter"], prev_v["verse"])] = prev_idx
                    parsed["verse"] = parsed["verse"] + "b"
                    if parsed["alt_v"] and not parsed["alt_v"].endswith(('a','b')):
                        parsed["alt_v"] = parsed["alt_v"] + "b"

                current = parsed
                state = "HEB"
            continue

        if current is None:
            continue

        if full_text == "Hebrew Transcription":
            state = "ENG"
            continue
        if full_text == "Interlinear Chart":
            state = "CHART"
            continue

        # ── Translation block ───────────────────────────────────────────────
        if full_text.startswith("Translation:"):
            eng_body_text, eng_markers = english_text_and_notes_from_block(block)
            eng_body_text, eng_markers = _strip_translation_prefix(eng_body_text, eng_markers)

            if "does not exist" in eng_body_text.lower():
                current["empty"] = True
                state = "ENG"
                continue
            for marker in ["The Scriptures:", "Aramaic:", "Greek:", "Does Not Exist"]:
                idx = eng_body_text.find(marker)
                if idx >= 0:
                    eng_markers = [(off, n) for off, n in eng_markers if off < idx]
                    eng_body_text = eng_body_text[:idx].strip()
            if eng_body_text:
                offset_shift = len(current["english"]) + (1 if current["english"] else 0)
                current["english"] += (" " if current["english"] else "") + eng_body_text
                current["eng_markers"].extend((off + offset_shift, n) for off, n in eng_markers)
            state = "ENG"
            continue

        # ── Content by state ────────────────────────────────────────────────
        if state == "HEB":
            lower = full_text.lower()
            if not current["empty"] and ("does not exist" in lower or
                                         "changes the order" in lower):
                current["empty"] = True
                state = "ENG"
                continue
            if has_hebrew(full_text) and is_wide(x0, x1):
                heb_text, heb_marks = hebrew_text_and_notes(block)
                offset_shift = len(current["hebrew"]) + (1 if current["hebrew"] else 0)
                current["hebrew"] += (" " if current["hebrew"] else "") + heb_text
                current["heb_markers"].extend((off + offset_shift, n) for off, n in heb_marks)

        elif state == "ENG":
            if is_wide(x0, x1) and not has_hebrew(full_text):
                skip_prefixes = ("The Scriptures:", "Aramaic:", "Greek:", "Does Not Exist",
                                 "Copyright", "The Scroll of Mysteries", "Interlinear")
                if not any(full_text.startswith(p) for p in skip_prefixes):
                    eng_text, eng_marks = english_text_and_notes_from_block(block)
                    offset_shift = len(current["english"]) + (1 if current["english"] else 0)
                    current["english"] += (" " if current["english"] else "") + eng_text
                    current["eng_markers"].extend((off + offset_shift, n) for off, n in eng_marks)

        elif state == "CHART":
            pass

    pending_notes.update(page_footnotes)

attach_pending_notes(current, pending_notes)
if current is not None:
    verses.append(current)

print(f"Parsed {len(verses)} verse records across {len(set(v['chapter'] for v in verses))} chapters.")
for v in verses[:3]:
    print(f"  Rev {v['chapter']}:{v['verse']} (alt:{v['alt_v']}) empty={v['empty']}")
    print(f"    heb: {clean_hebrew(v['hebrew'])[:80]}")
    print(f"    eng: {v['english'][:60]}")
    if v['heb_markers']:
        print(f"    heb_markers: {v['heb_markers'][:5]}")
    if v['eng_markers']:
        print(f"    eng_markers: {v['eng_markers'][:5]}")
    if v['notes']:
        print(f"    notes: {list(v['notes'].keys())[:8]}")

In [ ]:
# ── Inspect parsed verses ────────────────────────────────────────────────────
alt_verses = [v for v in verses if v['alt_v']]
empty_verses = [v for v in verses if v['empty']]
print(f"Alt-numbered verses: {len(alt_verses)}")
print(f"Empty placeholder verses: {len(empty_verses)}")
print("\nFirst 10 alt-numbered:")
for v in alt_verses[:10]:
    print(f"  Rev {v['chapter']}:{v['verse']}  Cochin {v['alt_ch']}:{v['alt_v']}")
print("\nEmpty verses:")
for v in empty_verses:
    print(f"  Rev {v['chapter']}:{v['verse']}")

# RTL spot-check
print("\n=== RTL spot-check ===")
v11 = next((v for v in verses if v['chapter']==1 and v['verse']=='1'), None)
if v11:
    heb = clean_hebrew(v11['hebrew'])
    expected_start = "אלה הסודות"
    ok = heb.startswith(expected_start)
    print(f"  Rev 1:1 starts with '{expected_start}': {ok}")
    print(f"  First 80 chars: {heb[:80]}")

# Verse 2:27a/27b check
print("\n=== Uppercase suffix check (2:27a/27b) ===")
for v in verses:
    if v['chapter']==2 and v['verse'] in ['27a','27b']:
        print(f"  Rev 2:{v['verse']} present, heb[:40]={clean_hebrew(v['hebrew'])[:40]}")

# Repeated verse check (2:21a/21b)
print("\n=== Repeated verse split (2:21a/21b) ===")
for v in verses:
    if v['chapter']==2 and v['verse'] in ['21a','21b','21']:
        print(f"  Rev 2:{v['verse']} present, heb[:40]={clean_hebrew(v['hebrew'])[:40]}")

In [ ]:
# ── OSIS building helpers ─────────────────────────────────────────────────────

OSIS_NS  = "http://www.bibletechnologies.net/2003/OSIS/namespace"
XSI_NS   = "http://www.w3.org/2001/XMLSchema-instance"
ALT_NS   = "https://projecttruthministries.org/studies/cochin-revelation/"
MS_NS    = "https://projecttruthministries.org/studies/cochin-revelation/"

SCHEMA_LOC = (f"{OSIS_NS} "
              "http://www.bibletechnologies.net/osisCore.2.1.1.xsd")

def make_osis_root(nsmap_extra=None):
    nsmap = {
        None:  OSIS_NS,
        "xsi": XSI_NS,
    }
    if nsmap_extra:
        nsmap.update(nsmap_extra)
    root = etree.Element(f"{{{OSIS_NS}}}osis", nsmap=nsmap)
    root.set(f"{{{XSI_NS}}}schemaLocation", SCHEMA_LOC)
    return root

def make_header_hebrew(parent):
    hdr = etree.SubElement(parent, f"{{{OSIS_NS}}}header")
    for osisWork, title, lang, extra in [
        ("MS.Oo.1.16.2_REV_Hebrew", "Revelation (Cochin MS Oo.1.16.2)", "he", [
            ("identifier", {"type": "OSIS"}, "MS.Oo.1.16.2_REV_Hebrew"),
            ("refSystem", {}, "MS.Oo.1.16.2"),
            ("language", {}, "he"),
        ]),
        ("bible", "Referenced versification (standard)", "he", [
            ("identifier", {"type": "OSIS"}, "bible"),
            ("refSystem", {}, "StandardV11N"),
            ("language", {}, "he"),
        ]),
        ("MS.Oo.1.16.2", "Revelation (Cochin MS Oo.1.16.2)", "he", [
            ("scope", {}, "REV"),
            ("type", {"type": "x-manuscript"}, "Manuscript"),
            ("identifier", {"type": "shelfmark"}, "MS.Oo.1.16.2"),
            ("identifier", {"type": "URI"}, ALT_NS),
            ("publisher", {}, "Project Truth Ministries"),
            ("contributor", {"role": "scr", "file-as": "Rahabi, Ezekiel"}, "Rabbi Ezekiel Ra\u1e25abi"),
            ("date", {"event": "original", "type": "ISO"}, "ca. 1730"),
            ("description", {}, (
                "The Cochin Hebrew New Testament manuscripts are significant 18th-century Hebrew "
                "versions of the New Testament currently housed at Cambridge. Digitized version "
                "of the MS Oo.1.16.2 can be found on the NLI website."
            )),
        ]),
    ]:
        w = etree.SubElement(hdr, f"{{{OSIS_NS}}}work", osisWork=osisWork)
        t = etree.SubElement(w, f"{{{OSIS_NS}}}title")
        t.text = title
        for tag, attrs, val in extra:
            el = etree.SubElement(w, f"{{{OSIS_NS}}}{tag}", **attrs)
            el.text = val

def make_header_translation(parent):
    hdr = etree.SubElement(parent, f"{{{OSIS_NS}}}header")
    w = etree.SubElement(hdr, f"{{{OSIS_NS}}}work", osisWork="MS.Oo.1.16.2_PTM")
    for tag, attrs, val in [
        ("title", {}, "Translation of Revelation (Cochin MS Oo.1.16.2)"),
        ("scope", {}, "REV"),
        ("type", {"type": "x-bible"}, "Edition"),
        ("creator", {"role": "trl"}, "Project Truth Ministries"),
        ("identifier", {"type": "URI"}, ALT_NS),
        ("contributor", {"role": "trc", "file-as": "Baca, Janice F."}, "Janice F. Baca"),
        ("date", {"event": "eversion", "type": "ISO"}, "2024"),
        ("rights", {}, "\u00a9 copyright 2024 Janice F. Baca"),
        ("description", {}, (
            "English translation of the Cochin MS Oo.1.16.2 Hebrew Revelation. "
            "Translated by Janice F. Baca, 2024."
        )),
    ]:
        el = etree.SubElement(w, f"{{{OSIS_NS}}}{tag}", **attrs)
        el.text = val

def verse_osisID(chapter, verse):
    return f"Rev.{chapter}.{verse}"

def alt_osisID(alt_ch, alt_v):
    if alt_ch is None or alt_v is None:
        return None
    return f"Rev.{alt_ch}.{alt_v}"

print("OSIS helpers defined.")

In [ ]:
# ── Generate Hebrew OSIS ──────────────────────────────────────────────────────

ALT_PREFIX = "{" + ALT_NS + "}"

def _interleave_notes(verse_el, text, markers, notes_dict, ns=OSIS_NS):
    """Place text and inline <note> elements using marker offsets."""
    sorted_marks = sorted(markers, key=lambda x: x[0])
    prev = 0
    last_el = None
    for offset, n in sorted_marks:
        n_str = str(n)
        if n_str not in notes_dict:
            continue
        chunk = text[prev:offset]
        if last_el is None:
            verse_el.text = (verse_el.text or "") + chunk
        else:
            last_el.tail = (last_el.tail or "") + chunk
        note_el = etree.SubElement(verse_el, f"{{{ns}}}note",
                                   type="footnote", n=n_str)
        note_el.text = notes_dict[n_str]
        note_el.tail = ""
        last_el = note_el
        prev = offset
    # remaining text
    remainder = text[prev:]
    if last_el is None:
        verse_el.text = (verse_el.text or "") + remainder
    else:
        last_el.tail = (last_el.tail or "") + remainder


def build_hebrew_osis(verses, with_notes=False):
    nsmap_extra = {"alt": ALT_NS, "ms": MS_NS}
    root = make_osis_root(nsmap_extra)

    work_id = "MS.Oo.1.16.2_REV_Hebrew" + ("_Commented" if with_notes else "")
    osisText = etree.SubElement(root, f"{{{OSIS_NS}}}osisText",
                                osisIDWork=work_id,
                                osisRefWork="bible")
    osisText.set("{http://www.w3.org/XML/1998/namespace}lang", "he")

    make_header_hebrew(osisText)

    div_book = etree.SubElement(osisText, f"{{{OSIS_NS}}}div",
                                type="book", osisID="Rev")

    current_chapter = None
    chapter_el = None

    for v in verses:
        ch = v["chapter"]
        if ch != current_chapter:
            chapter_el = etree.SubElement(div_book, f"{{{OSIS_NS}}}chapter",
                                          osisID=f"Rev.{ch}")
            current_chapter = ch

        osisID  = verse_osisID(ch, v["verse"])
        n_val   = v["verse"]
        alt_id  = alt_osisID(v["alt_ch"], v["alt_v"])

        attrs = {"osisID": osisID, "n": n_val}
        if alt_id:
            attrs[f"{ALT_PREFIX}num"] = alt_id

        verse_el = etree.SubElement(chapter_el, f"{{{OSIS_NS}}}verse", **attrs)

        if not v["empty"]:
            hebrew_text = clean_hebrew(v["hebrew"])
            if with_notes and v["notes"] and v["heb_markers"]:
                _interleave_notes(verse_el, hebrew_text, v["heb_markers"], v["notes"])
            elif with_notes and v["notes"]:
                verse_el.text = hebrew_text + " "
                for n_str, note_text in sorted(v["notes"].items(), key=lambda x: int(x[0])):
                    note_el = etree.SubElement(verse_el, f"{{{OSIS_NS}}}note",
                                               type="footnote", n=n_str)
                    note_el.text = note_text
                    note_el.tail = " "
            else:
                verse_el.text = hebrew_text

    return root

heb_tree      = build_hebrew_osis(verses, with_notes=False)
heb_commented = build_hebrew_osis(verses, with_notes=True)
print("Hebrew OSIS trees built.")

In [ ]:
# ── Generate Translation OSIS ─────────────────────────────────────────────────

def build_translation_osis(verses):
    root = make_osis_root()

    osisText = etree.SubElement(root, f"{{{OSIS_NS}}}osisText",
                                osisIDWork="REV",
                                osisRefWork="bible")
    osisText.set("{http://www.w3.org/XML/1998/namespace}lang", "en")

    make_header_translation(osisText)

    current_chapter = None
    chapter_el = None

    for v in verses:
        ch = v["chapter"]
        if ch != current_chapter:
            chapter_el = etree.SubElement(osisText, f"{{{OSIS_NS}}}div",
                                          type="chapter",
                                          osisID=f"Rev.{ch}")
            current_chapter = ch

        osisID = verse_osisID(ch, v["verse"])
        verse_el = etree.SubElement(chapter_el, f"{{{OSIS_NS}}}verse",
                                    osisID=osisID)

        if not v["empty"] and v["english"]:
            eng_text = v["english"].replace("\n", " ").strip()
            if v["notes"] and v["eng_markers"]:
                _interleave_notes(verse_el, eng_text, v["eng_markers"], v["notes"])
            elif v["notes"]:
                verse_el.text = eng_text + " "
                for n_str, note_text in sorted(v["notes"].items(), key=lambda x: int(x[0])):
                    note_el = etree.SubElement(verse_el, f"{{{OSIS_NS}}}note",
                                               type="footnote", n=n_str)
                    note_el.text = note_text
                    note_el.tail = " "
            else:
                verse_el.text = eng_text

    return root

trans_tree = build_translation_osis(verses)
print("Translation OSIS tree built.")

In [ ]:
# ── Write output files ────────────────────────────────────────────────────────

outputs = [
    (heb_tree,      f"{STEM}_hebrew.osis"),
    (heb_commented, f"{STEM}_hebrew_commented.osis"),
    (trans_tree,    f"{STEM}_translation.osis"),
]

for tree, filename in outputs:
    out_path = OUT_DIR / filename
    tree_obj = etree.ElementTree(tree)
    tree_obj.write(
        str(out_path),
        xml_declaration=True,
        encoding="UTF-8",
        pretty_print=True,
    )
    size_kb = out_path.stat().st_size // 1024
    print(f"Wrote {out_path}  ({size_kb} KB)")

In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────
from collections import Counter

chapter_counts = Counter(v['chapter'] for v in verses)
print("=== Verse count by chapter ===")
for ch in sorted(chapter_counts):
    print(f"  Rev {ch}: {chapter_counts[ch]} verses")

alt_verses   = [v for v in verses if v['alt_v']]
empty_verses = [v for v in verses if v['empty']]
no_heb       = [v for v in verses if not v['empty'] and not v['hebrew'].strip()]
print(f"\nAlt-numbered: {len(alt_verses)},  Empty: {len(empty_verses)},  No Hebrew: {len(no_heb)}")
if no_heb:
    print("Verses missing Hebrew text:")
    for v in no_heb:
        print(f"  Rev {v['chapter']}:{v['verse']}")

print("\n=== Sample Rev 1:1 ===")
v11 = next((v for v in verses if v['chapter']==1 and v['verse']=='1'), None)
if v11:
    print(f"  heb: {clean_hebrew(v11['hebrew'])[:80]}")
    print(f"  eng: {v11['english'][:80]}")

print("\n=== Alt-numbered sample (Rev 2:6-9) ===")
for v in verses:
    if v['chapter']==2 and v['verse'] in ['6','7','8','9']:
        print(f"  Rev 2:{v['verse']}  Cochin {v['alt_ch']}:{v['alt_v']}  empty={v['empty']}  heb:{clean_hebrew(v['hebrew'])[:40]}")

print("\n=== XML well-formedness check ===")
for tree, filename in outputs:
    try:
        etree.tostring(tree, encoding="unicode")
        print(f"  {filename}: OK")
    except Exception as e:
        print(f"  {filename}: ERROR — {e}")